# Try Fungen Operator on Minerva

Use this guide from a Jupyter terminal or login node to install a test copy, start a local model, and ask the host Codex to set up xQTL through fine-mapping. You need Singularity, access to LSF GPU queues, and a Python 3 notebook kernel for the cells below. Another site can use Apptainer and its own scheduler and storage.

The saved execution covers only the variable definitions and the two read-only help commands. **Setup, serve, Codex, xQTL and check cells are intentionally unexecuted.** Run interactive commands in a terminal; this notebook records the commands without opening interactive sessions.

Follow the steps in order: choose paths, inspect help, install, start the server and Codex, try xQTL, then check the installation. See the [README](../README.md) and [container validation](../container/VALIDATION.md) for the distribution contract and recorded acceptance.

## 1. Set site values

These are Minerva worked examples. On another site, change `IMAGE`, `INSTALL`, `MODELS` and `XQTL_ROOT` to accessible paths, set `RUNTIME` to `apptainer` if appropriate, and set `SCHEDULER` to `slurm` or `local`. Choose your own writable install and xQTL folders. The scratch install keeps home clean.

The first cell exports values to subsequent `%%bash` cells and prints the equivalent shell declarations. **Copy its output into your terminal first** so terminal commands use the same values. Every later command uses these variables. If you change this cell, rerun it and copy the updated declarations.

In [1]:
import os
import shlex

IMAGE = "/sc/arion/projects/load/users/sunh14/OLLAMA/images/round2/current/fungen-operator-ollama0.40.6-codex0.155.1.sif"
INSTALL = "/sc/arion/scratch/sunh14/fungen_try"
MODELS = "/sc/arion/scratch/sunh14/fungen_stranger/round2/xs-models"
XQTL_ROOT = "/sc/arion/scratch/sunh14/xqtl_try"
RUNTIME = "singularity"
SCHEDULER = "lsf"
MODEL = "laguna-xs-2.1:q4_K_M"

for name in ("IMAGE", "INSTALL", "MODELS", "XQTL_ROOT", "RUNTIME", "SCHEDULER", "MODEL"):
    value = globals()[name]
    os.environ[name] = value
    print(f"export {name}={shlex.quote(value)}")

export IMAGE=/sc/arion/projects/load/users/sunh14/OLLAMA/images/round2/current/fungen-operator-ollama0.40.6-codex0.155.1.sif
export INSTALL=/sc/arion/scratch/sunh14/fungen_try
export MODELS=/sc/arion/scratch/sunh14/fungen_stranger/round2/xs-models
export XQTL_ROOT=/sc/arion/scratch/sunh14/xqtl_try
export RUNTIME=singularity
export SCHEDULER=lsf
export MODEL=laguna-xs-2.1:q4_K_M


Laguna XS Q4 is the quickest option and uses one GPU. The reused acceptance-test XS store occupies about 19 GB. Scratch can be purged: if that store disappears, choose download **yes** with your own writable model folder; the download is about 20 GB and needs additional free space and quota.

For Laguna S, change `MODEL` in the first cell to `laguna-s-2.1:q4_K_M` and `MODELS` to `/sc/arion/projects/load/users/sunh14/OLLAMA/staging_models`, then rerun the cell. This option uses two GPUs. The validation record notes that this staged alias actually contains a Q8_0 payload; it is an existing reused store.

## 2. Inspect the image interface

These two commands only print help. Their saved outputs describe the delivered image, which may differ from newer source changes.

In [2]:
%%bash
set -euo pipefail
"$RUNTIME" run "$IMAGE" --help

usage: fungen.py [-h] {setup,serve,download,codex,check,xqtl} ...

Install on the host; serve Laguna

 in Apptainer; run host Codex.

positional arguments:
  {setup,serve,download,codex,check,xqtl}
    

setup               Install host tools and configure a backend
    serve               Start the GPU

 server
    download            Image only: acquire/resume model blobs
    codex               Host 

only: wait for the server, then run Codex
    check               Report each installation check and

 its exact fix
    xqtl                Host only: guided xQTL setup and synthetic test

options:
  -

h, --help            show this help message and exit


In [3]:
%%bash
set -euo pipefail
"$RUNTIME" run "$IMAGE" setup --help

usage: fungen.py setup [-h] [--non-interactive] [--install INSTALL] [--backend {local,openai}] [--sc

heduler {lsf,slurm,local}] [--model {laguna-xs-2.1:q4_K_M,laguna-xs-2.1:q8_0,laguna-s-2.1:q4_K_M,lag

una-s-2.1:q8_0}]
                       [--models-dir MODELS_DIR] [--download {yes,no}] [--context {

32768,262144}] [--add-path {yes,no}] [--image IMAGE] [--runtime RUNTIME] [--account ACCOUNT] [--gpu-

queue GPU_QUEUE] [--port PORT] [--offline-check]

options:
  -h, --help            show this help me

ssage and exit
  --non-interactive
  --install INSTALL
  --backend {local,openai}
  --scheduler {lsf

,slurm,local}
  --model {laguna-xs-2.1:q4_K_M,laguna-xs-2.1:q8_0,laguna-s-2.1:q4_K_M,laguna-s-2.1:q8

_0}
  --models-dir MODELS_DIR
  --download {yes,no}
  --context {32768,262144}
  --add-path {yes,no}


  --image IMAGE
  --runtime RUNTIME
  --account ACCOUNT
  --gpu-queue GPU_QUEUE
  --port PORT
  --o

ffline-check       Report offline guidance without a network probe


## 3. Install from a terminal

After copying the declarations above into a terminal, run:

```bash
"$RUNTIME" run "$IMAGE"
```

The setup prompt list below is copied exactly from the README. Defaults describe the installer's choices; use the suggested answers in the table.

```text
Install folder [~/.fungen]:
Scheduler (lsf/slurm/local) [detected scheduler]:
Model backend (local/openai) [local]:
Models (download):
  laguna-xs-2.1:q4_K_M: 20 GB
  laguna-xs-2.1:q8_0: 36 GB
  laguna-s-2.1:q4_K_M: 96 GB
  laguna-s-2.1:q8_0: 128 GB
Model tag [largest choice fitting visible GPU memory; XS Q4 when unknown]:
Download model? (yes/no) [no]:
Model folder to download into or reuse [scratch/project on clusters; ~/.fungen/models locally]:
Context (32768; advanced 262144 needs extra GPU memory) [32768]:
Add the install bin folder to your shell PATH? (yes/no) [no]:
```

| Prompt | Suggested answer |
|---|---|
| Install folder | Value of `INSTALL` |
| Scheduler | Value of `SCHEDULER` (`lsf` here) |
| Model backend | `local` |
| Model tag | Value of `MODEL` (XS Q4 here) |
| Download model? | `no` to reuse the existing store |
| Model folder to download into or reuse | Value of `MODELS` |
| Context | `32768` |
| Add the install bin folder to your shell PATH? | `no` |

At a prompt, paste the actual path or value printed by the first cell: shell variables are not expanded by interactive prompts.

Alternatively, the following non-interactive cell supplies the same answers. Choose either terminal setup or this equivalent cell.

In [ ]:
%%bash
set -euo pipefail
"$RUNTIME" run "$IMAGE" setup --non-interactive   --install "$INSTALL" --scheduler "$SCHEDULER" --backend local   --model "$MODEL" --download no --models-dir "$MODELS"   --context 32768 --add-path no

## 4. Start the model and host Codex

Run each command in your terminal, in order. `serve` submits the GPU server as an LSF job. `codex` waits for the recorded server endpoint and opens host Codex with the operator loaded. With PATH unchanged, use the full launcher path as shown.

In [ ]:
%%bash
set -euo pipefail
"$INSTALL/bin/fungen" serve

In [ ]:
%%bash
set -euo pipefail
"$INSTALL/bin/fungen" codex

## 5. Try the first task

In the host Codex session, say:

> set up the xQTL protocol through fine-mapping

The operator asks before each step. Choose **mamba** for software setup: Pixi currently stops at the upstream `pixi-setup` jq defect documented in validation.

As a direct alternative, run the next command from a terminal. It uses your chosen xQTL root and creates the guided setup notebook. Review and confirm each step; the prepared synthetic test precedes your own harmonized-data analysis.

In [ ]:
%%bash
set -euo pipefail
"$INSTALL/bin/fungen" xqtl --root "$XQTL_ROOT"

## 6. Check and finish

After setup and server startup, run the check command. Each item reports pass/fail and gives the exact fix. With PATH disabled, its PATH check may need the explicit export suggested by the command; the full launcher path remains usable.

In [ ]:
%%bash
set -euo pipefail
"$INSTALL/bin/fungen" check

The installation lives in the chosen install folder plus `~/.config/fungen/config.toml`. The user's own Codex configuration stays untouched; local sessions use the install's separate Codex home. Reused model stores and the chosen xQTL working folder are separate storage locations, and any downloaded models stay in the chosen model folder. Review your server's native job log and stop your own server job when finished before removing an installation. This guide performs no cleanup.

**OpenAI alternative:** choose backend `openai` during setup; model questions are skipped. Skip `serve`, authenticate using `codex login --device-auth` or an `OPENAI_API_KEY` supplied in your terminal environment, then use the same host Codex launcher. Keep credentials in your terminal environment and out of notebook cells and saved outputs.

**Path exercise:** before setup at another site, identify its image location, writable install and analysis folders, model store, runtime and scheduler. Update the first cell and copy its new shell declarations into your terminal.